# Model Training — Encrypted Malicious Traffic Detection
Trains LSTM, ResNet-34, XGBoost (Layer-1) and RF/Average ensemble (Layer-2).
Assumes preprocessed tensors are in `tensors/`, scalers in `scalers/`.

In [1]:
!pip install -q tensorflow torch torchvision xgboost scikit-learn joblib

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


In [3]:
import platform
import os
from google.colab import drive
drive.mount('/content/drive')

print("OS:", platform.system())
print("Working Directory:", os.getcwd())

ModuleNotFoundError: No module named 'google'

In [14]:
import os, gc, numpy as np, joblib
os.makedirs('models', exist_ok=True)
os.makedirs('probs', exist_ok=True)

# Set working directory to your repo root
# os.chdir('/kaggle/working/')  # uncomment for Kaggle
# os.chdir('/content/drive/MyDrive/...')  # uncomment for Drive

## 1. LSTM Training (TensorFlow/Keras)

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np

X_train = np.load('tensors/X_lstm_train.npy')  # (N, 15, 85)
X_val   = np.load('tensors/X_lstm_val.npy')
X_test  = np.load('tensors/X_lstm_test.npy')
y_train = np.load('tensors/y_train.npy')
y_val   = np.load('tensors/y_val.npy')

print(f'LSTM shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Architecture from paper Section 5.1
model = Sequential([
    LSTM(128, input_shape=(15, 85), return_sequences=True),
    Dropout(0.3),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dense(2, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
ckpt = ModelCheckpoint('models/lstm_model.h5', monitor='val_accuracy', save_best_only=True, mode='max')
es   = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100, batch_size=32,
    callbacks=[es, ckpt], verbose=2
)

In [ ]:
# Save probabilities for Layer-2 (train + val + test)
np.save('probs/lstm_probs_train.npy', model.predict(X_train))
np.save('probs/lstm_probs_val.npy',   model.predict(X_val))
np.save('probs/lstm_probs_test.npy',  model.predict(X_test))
print('LSTM probs saved')

del X_train, X_val, X_test
gc.collect()

## 2. ResNet-34 Training (PyTorch)

In [ ]:
import torch, torch.nn as nn, torchvision.models as models
from torch.utils.data import TensorDataset, DataLoader

X_train = np.load('tensors/X_resnet_train.npy')  # (N, 1, 38, 38)
X_val   = np.load('tensors/X_resnet_val.npy')
X_test  = np.load('tensors/X_resnet_test.npy')
y_train = np.load('tensors/y_train.npy')
y_val   = np.load('tensors/y_val.npy')

print(f'ResNet shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')

train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).long())
val_ds   = TensorDataset(torch.from_numpy(X_val).float(),   torch.from_numpy(y_val).long())
test_ds  = TensorDataset(torch.from_numpy(X_test).float(),  torch.zeros(len(X_test)).long())
loader_train = DataLoader(train_ds, batch_size=32, shuffle=True)
loader_val   = DataLoader(val_ds,   batch_size=32)
loader_test  = DataLoader(test_ds,  batch_size=32)

In [ ]:
# ResNet-34 adapted for 1-channel 38x38 input (paper Section 5.2)
net = models.resnet34(pretrained=False)
net.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
net.maxpool = nn.Identity()
net.fc = nn.Linear(512, 2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
print(f'Using device: {device}')

In [ ]:
def train_one_epoch():
    net.train()
    loss_sum = 0.0
    for xb, yb in loader_train:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(net(xb), yb)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()
    return loss_sum / len(loader_train)

def get_probs(loader):
    net.eval()
    all_p = []
    with torch.no_grad():
        for xb, *_ in loader:
            logits = net(xb.to(device))
            all_p.append(torch.softmax(logits, 1).cpu().numpy())
    return np.concatenate(all_p)

def eval_acc():
    net.eval()
    c, t = 0, 0
    with torch.no_grad():
        for xb, yb in loader_val:
            preds = net(xb.to(device)).argmax(1).cpu()
            c += (preds == yb).sum().item(); t += yb.size(0)
    return c / t

best_acc, patience, wait = 0.0, 5, 0
for epoch in range(1, 101):
    tloss = train_one_epoch()
    vacc  = eval_acc()
    print(f'Epoch {epoch:03d} | loss {tloss:.4f} | val_acc {vacc:.4f}')
    if vacc > best_acc:
        best_acc = vacc
        torch.save(net.state_dict(), 'models/resnet34.pt'); wait = 0
    else:
        wait += 1
        if wait >= patience: print('Early stopping'); break

In [ ]:
# Reload best model and save all probabilities
net.load_state_dict(torch.load('models/resnet34.pt', map_location=device))
np.save('probs/resnet_probs_train.npy', get_probs(loader_train))
np.save('probs/resnet_probs_val.npy',   get_probs(loader_val))
np.save('probs/resnet_probs_test.npy',  get_probs(loader_test))
print('ResNet probs saved')

del X_train, X_val, X_test, train_ds, val_ds, test_ds
gc.collect()

## 3. XGBoost Training

In [ ]:
import xgboost as xgb

X_train = np.load('tensors/X_xgb_train.npy')  # (N, 97)
X_val   = np.load('tensors/X_xgb_val.npy')
X_test  = np.load('tensors/X_xgb_test.npy')
y_train = np.load('tensors/y_train.npy')
y_val   = np.load('tensors/y_val.npy')

print(f'XGBoost shapes: train={X_train.shape}, val={X_val.shape}, test={X_test.shape}')

# Hyperparameters from paper Section 5.3
xgb_clf = xgb.XGBClassifier(
    base_score=0.5, learning_rate=0.05,
    n_estimators=100, max_depth=10,
    objective='binary:logistic', eval_metric='logloss',
    use_label_encoder=False, n_jobs=-1, random_state=42
)
xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)],
            early_stopping_rounds=10, verbose=False)
joblib.dump(xgb_clf, 'models/xgb_model.pkl')

np.save('probs/xgb_probs_train.npy', xgb_clf.predict_proba(X_train))
np.save('probs/xgb_probs_val.npy',   xgb_clf.predict_proba(X_val))
np.save('probs/xgb_probs_test.npy',  xgb_clf.predict_proba(X_test))
print('XGBoost probs saved')

del X_train, X_val, X_test
gc.collect()

## 4. Layer-2 Ensemble

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train RF on TRAIN-split probabilities (no data leakage — plan Section 6.1)
lstm_tr   = np.load('probs/lstm_probs_train.npy')
resnet_tr = np.load('probs/resnet_probs_train.npy')
xgb_tr    = np.load('probs/xgb_probs_train.npy')
X_l2_train = np.hstack([lstm_tr, resnet_tr, xgb_tr])  # (N_train, 6)
y_train = np.load('tensors/y_train.npy')

rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, max_features='sqrt',
    bootstrap=True, min_samples_split=2, n_jobs=-1, random_state=42
)
rf.fit(X_l2_train, y_train)
joblib.dump(rf, 'models/rf_layer2.pkl')
print('RF Layer-2 trained on train probs')

In [ ]:
# Evaluate on VALIDATION split
lstm_v   = np.load('probs/lstm_probs_val.npy')
resnet_v = np.load('probs/resnet_probs_val.npy')
xgb_v    = np.load('probs/xgb_probs_val.npy')
X_l2_val = np.hstack([lstm_v, resnet_v, xgb_v])
y_val    = np.load('tensors/y_val.npy')

avg_val = X_l2_val.reshape(-1, 3, 2).mean(axis=1)

def show_metrics(name, probs, y_true):
    preds = np.argmax(probs, axis=1)
    print(f'\n=== {name} ===')
    print(classification_report(y_true, preds, target_names=['Benign','Malicious']))
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    print(f'FPR: {fp/(fp+tn):.4%}   AUC: {roc_auc_score(y_true, probs[:,1]):.4%}')

show_metrics('RF Layer-2 (Val)',       rf.predict_proba(X_l2_val), y_val)
show_metrics('Average Ensemble (Val)', avg_val, y_val)

## 5. Final Test-Set Evaluation

In [ ]:
lstm_te   = np.load('probs/lstm_probs_test.npy')
resnet_te = np.load('probs/resnet_probs_test.npy')
xgb_te    = np.load('probs/xgb_probs_test.npy')
X_l2_test = np.hstack([lstm_te, resnet_te, xgb_te])
y_test    = np.load('tensors/y_test.npy')

avg_test = X_l2_test.reshape(-1, 3, 2).mean(axis=1)

show_metrics('RF Layer-2 (Test)',       rf.predict_proba(X_l2_test), y_test)
show_metrics('Average Ensemble (Test)', avg_test, y_test)